# SIMBES — Módulo 7: Confiabilidad y MTBF

**Objetivo:** Implementar y visualizar los conceptos de confiabilidad para sistemas BES/ESP:  
1. Función de supervivencia R(t) = e^(−t/MTBF)  
2. Resultado fundamental: R(MTBF) = e⁻¹ ≈ 36.77%  
3. Estimación MTBF MLE con datos censurados  
4. Intervalos de confianza Chi² (Nelson 1982)  
5. Comparación con benchmarks por ambiente  

---
**Fuentes:**  
- Nelson, W. (1982). Applied Life Data Analysis. Wiley.  
- Meeker & Escobar (1998). Statistical Methods for Reliability Data.  
- mtbf-benchmarks.json — datos pedagógicos SIMBES

In [ ]:
import sys
import os
import json
import math
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#0B0F1A',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1E293B',
    'axes.labelcolor':  '#CBD5E1',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'text.color':       '#CBD5E1',
    'grid.color':       '#1E293B',
    'grid.linestyle':   '--',
    'legend.facecolor': '#111827',
    'legend.edgecolor': '#1E293B',
    'font.family':      'monospace',
})

with open('../frontend/src/data/mtbf-benchmarks.json', 'r', encoding='utf-8') as f:
    BENCH_DATA = json.load(f)
BENCHMARKS = BENCH_DATA['benchmarks']
BENCH_COLORS = ['#22C55E', '#38BDF8', '#F59E0B', '#EF4444']

print("Setup OK ✓")

## 1. Función de Supervivencia Exponencial

$$
R(t) = e^{-t/\text{MTBF}} = e^{-\lambda t}
$$

**Resultado fundamental:**
$$
R(\text{MTBF}) = e^{-1} \approx 0.3679 = 36.77\%
$$

Al cumplir el MTBF nominal, solo el **36.77%** de los equipos sigue operativo.

In [ ]:
def R(t, MTBF):
    """Función de supervivencia exponencial."""
    return np.exp(-t / MTBF)

# Verificación del resultado fundamental
R_at_MTBF = math.exp(-1)
print(f"R(MTBF) = e⁻¹ = {R_at_MTBF:.6f} = {R_at_MTBF*100:.4f}%")
print(f"Mediana = MTBF × ln(2) = MTBF × {math.log(2):.4f}")
print(f"Si MTBF=913d → mediana = {913*math.log(2):.0f} días")

# Curvas para los 4 ambientes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

t_arr = np.linspace(0, 4000, 500)

for bench, color in zip(BENCHMARKS, BENCH_COLORS):
    MTBF_d = bench['MTBF_days']
    R_arr  = R(t_arr, MTBF_d)
    ax1.plot(t_arr, R_arr * 100, color=color, lw=2,
             label=f"{bench['environment'].split()[0]} {bench['environment'].split()[1] if len(bench['environment'].split())>1 else ''} (MTBF={MTBF_d}d)")
    # Marcar MTBF en la curva
    ax1.plot(MTBF_d, R_at_MTBF * 100, 'o', color=color, markersize=6, markeredgecolor='white', markeredgewidth=1)

ax1.axhline(R_at_MTBF * 100, color='#FB923C', lw=1.5, ls='--', label=f'R(MTBF) = 36.77%')
ax1.axhline(50, color='#64748B', lw=1, ls=':', label='50% (mediana)')
ax1.set_xlabel('Tiempo t [días]')
ax1.set_ylabel('R(t) — Probabilidad de supervivencia [%]')
ax1.set_title('Curvas R(t) por Ambiente Operativo', pad=10)
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 4000)
ax1.set_ylim(0, 100)

# Panel derecho: R(1 año) por ambiente
R_1y = [R(365, b['MTBF_days']) * 100 for b in BENCHMARKS]
names = [b['environment'].split()[0] for b in BENCHMARKS]
bars  = ax2.bar(names, R_1y, color=BENCH_COLORS, alpha=0.85, width=0.5)
ax2.axhline(R_at_MTBF * 100, color='#FB923C', lw=1.5, ls='--', label='36.77%')
ax2.axhline(50, color='#64748B', lw=1, ls=':', label='50%')

for bar, val in zip(bars, R_1y):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%',
             ha='center', fontsize=9, color='#CBD5E1')

ax2.set_xlabel('Ambiente')
ax2.set_ylabel('R(365 días) [%]')
ax2.set_title('Supervivencia al Año 1 por Ambiente', pad=10)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 110)

plt.suptitle('Función de Supervivencia Exponencial — BES', color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 2. MTBF MLE con Datos Censurados

$$
\hat{\text{MTBF}} = \frac{T_{\text{total}}}{r} = \frac{\sum t_{\text{fallas}} + \sum t_{\text{censurados}}}{r}
$$

Donde $r$ = número de fallas (no incluye censurados en el denominador).

**Sesgo de sobrevivencia inverso:** ignorar los censurados subestima el MTBF.

In [ ]:
def mtbf_mle(failure_times, censored_times):
    """
    Estimador MLE del MTBF para distribución exponencial con datos censurados.
    
    MTBF = T_total / r
    T_total = Σ t_fallas + Σ t_censurados
    
    @ref Nelson (1982). Applied Life Data Analysis. §3.
    """
    r       = len(failure_times)
    T_fail  = sum(failure_times)
    T_cens  = sum(censored_times)
    T_total = T_fail + T_cens
    
    if r == 0:
        return {'MTBF': float('inf'), 'r': 0, 'T_total': T_total, 'bias': True}
    
    MTBF_biased = T_fail / r        # ignora censurados ← ERROR
    MTBF_mle    = T_total / r       # correcto
    
    return {
        'MTBF_MLE':    round(MTBF_mle, 1),
        'MTBF_biased': round(MTBF_biased, 1),
        'r':           r,
        'n_total':     r + len(censored_times),
        'T_total':     T_total,
        'T_fail':      T_fail,
        'T_cens':      T_cens,
        'sesgo_pct':   round((MTBF_mle - MTBF_biased) / MTBF_mle * 100, 1),
    }

# Campo Colibrí: 4 fallas + 14 censurados
failure_times  = [95, 210, 340, 290]
censored_times = [300] * 14

result = mtbf_mle(failure_times, censored_times)
print("Campo Colibrí — Estimación MTBF")
print("-" * 45)
for k, v in result.items():
    print(f"  {k:<18}: {v}")

print(f"\nError por ignorar censurados: {result['sesgo_pct']}% de subestimación")

## 3. Intervalos de Confianza Chi² (Nelson 1982)

$$
\text{MTBF}_{\text{lower}} = \frac{2T}{\chi^2_{1-\alpha/2,\; 2r+2}}
\qquad
\text{MTBF}_{\text{upper}} = \frac{2T}{\chi^2_{\alpha/2,\; 2r}}
$$

Para IC 90% ($\alpha = 0.10$): quantiles $\chi^2_{0.95, 2r+2}$ y $\chi^2_{0.05, 2r}$.

In [ ]:
def mtbf_chi2_ci(T_total, r, alpha=0.10):
    """
    Intervalo de confianza Chi² para MTBF exponencial.
    
    MTBF_lower = 2T / χ²(1−α/2, 2r+2)
    MTBF_upper = 2T / χ²(α/2, 2r)
    
    @ref Nelson (1982). Applied Life Data Analysis. Eq. 3.3.
    """
    if r == 0:
        return {'lower': 0, 'upper': float('inf')}
    
    chi2_upper = stats.chi2.ppf(1 - alpha/2, 2*r + 2)   # denominador MTBF_lower
    chi2_lower = stats.chi2.ppf(alpha/2,     2*r)        # denominador MTBF_upper
    
    MTBF_lower = (2 * T_total) / chi2_upper
    MTBF_upper = (2 * T_total) / chi2_lower if chi2_lower > 0 else float('inf')
    
    return {
        'MTBF_lower':    round(MTBF_lower, 1),
        'MTBF_upper':    round(MTBF_upper, 1),
        'confidence_pct': round((1-alpha)*100),
        'chi2_upper':    round(chi2_upper, 2),
        'chi2_lower':    round(chi2_lower, 2),
    }

# Caso Campo Colibrí
T_total  = result['T_total']
r        = result['r']
MTBF_mle = result['MTBF_MLE']

print(f"T_total = {T_total} días · r = {r} fallas · MTBF_MLE = {MTBF_mle} días\n")
for alpha in [0.20, 0.10, 0.05]:
    ic = mtbf_chi2_ci(T_total, r, alpha)
    print(f"IC {ic['confidence_pct']}%: [{ic['MTBF_lower']}, {ic['MTBF_upper']}] días")

# ── Efecto del número de fallas sobre el IC ──────────────────────
r_range    = range(1, 31)
T_per_unit = MTBF_mle   # asumimos T_total = r × MTBF_mle

widths = []
lowers = []
uppers = []
for r_test in r_range:
    T_test = r_test * T_per_unit
    ic     = mtbf_chi2_ci(T_test, r_test, 0.10)
    lowers.append(ic['MTBF_lower'])
    uppers.append(min(ic['MTBF_upper'], T_per_unit * 10))

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(list(r_range), lowers, uppers, alpha=0.3, color='#FB923C', label='IC 90%')
ax.axhline(T_per_unit, color='#38BDF8', lw=2, ls='--', label=f'MTBF_MLE = {T_per_unit} días')
ax.plot(list(r_range), lowers, color='#22C55E', lw=2, label='IC_lower (accionable)')
ax.plot(list(r_range), uppers, color='#F59E0B', lw=2, ls='--', label='IC_upper')
ax.axvline(r, color='#EF4444', lw=1.5, ls=':', label=f'r = {r} (caso actual)')

ax.set_xlabel('Número de fallas observadas (r)')
ax.set_ylabel('MTBF [días]')
ax.set_title('Ancho del IC 90% vs Número de Fallas\n(MTBF_MLE constante)', pad=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, 30)
ax.set_ylim(0, T_per_unit * 5)
plt.tight_layout()
plt.show()

## 4. Impacto de Usar IC_lower vs MLE en Planificación

Comparación de decisiones de repuesto según el escenario MTBF elegido.

In [ ]:
ic_90 = mtbf_chi2_ci(T_total, r, 0.10)
MTBF_lower = ic_90['MTBF_lower']
MTBF_upper = ic_90['MTBF_upper']

N_EQUIPOS  = 18
t_arr_plan = np.linspace(0, 1500, 400)

R_mle   = R(t_arr_plan, MTBF_mle)
R_lower = R(t_arr_plan, MTBF_lower)
R_upper = R(t_arr_plan, min(MTBF_upper, 10000))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Curvas R(t)
ax1.fill_between(t_arr_plan, R_lower * 100, R_upper * 100, alpha=0.2, color='#FB923C', label='Rango IC 90%')
ax1.plot(t_arr_plan, R_mle   * 100, color='#38BDF8', lw=2.5, label=f'MTBF_MLE = {MTBF_mle} d')
ax1.plot(t_arr_plan, R_lower * 100, color='#22C55E', lw=2,   ls='--', label=f'IC_lower = {MTBF_lower} d')
ax1.axhline(36.77, color='#FB923C', lw=1, ls='--', alpha=0.7)
ax1.axvline(365,   color='#64748B', lw=1, ls=':', label='Año 1')
ax1.axvline(730,   color='#64748B', lw=1, ls=':', label='Año 2')

ax1.set_xlabel('t [días]')
ax1.set_ylabel('R(t) [%]')
ax1.set_title('R(t) — MLE vs IC_lower\nCampo Colibrí', pad=10)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 100)

# Fallas esperadas por año
anos = [1, 2, 3]
dias = [365 * a for a in anos]

x = np.arange(len(anos))
w = 0.3

fallas_mle   = [math.ceil(N_EQUIPOS * (1 - R(d, MTBF_mle)))   for d in dias]
fallas_lower = [math.ceil(N_EQUIPOS * (1 - R(d, MTBF_lower))) for d in dias]

ax2.bar(x - w/2, fallas_mle,   width=w, color='#38BDF8', alpha=0.85, label=f'MTBF_MLE={MTBF_mle}d', radius=4)
ax2.bar(x + w/2, fallas_lower, width=w, color='#22C55E', alpha=0.85, label=f'IC_lower={MTBF_lower}d', radius=4)

for i, (f_mle, f_low) in enumerate(zip(fallas_mle, fallas_lower)):
    ax2.text(i - w/2, f_mle + 0.2, str(f_mle), ha='center', fontsize=9, color='#38BDF8')
    ax2.text(i + w/2, f_low + 0.2, str(f_low), ha='center', fontsize=9, color='#22C55E')

ax2.set_xticks(x)
ax2.set_xticklabels([f'Año {a}' for a in anos])
ax2.set_ylabel(f'Fallas esperadas (de {N_EQUIPOS} equipos)')
ax2.set_title('Repuestos necesarios por escenario', pad=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Planificación de Mantenimiento — Campo Colibrí', color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print("\nResumen de planificación:")
print(f"{'Año':<5} {'MTBF_MLE':>10} {'IC_lower':>10}")
for a, f_m, f_l in zip(anos, fallas_mle, fallas_lower):
    print(f"Año {a}  {f_m:>10} eq.  {f_l:>8} eq.")

## 5. Resumen — Reglas de Confiabilidad BES

| Concepto | Valor / Fórmula |
|----------|----------------|
| R(MTBF) | e⁻¹ ≈ **36.77%** — siempre, para cualquier MTBF |
| Mediana | 0.693 × MTBF — tiempo hasta que falla el 50% |
| MTBF MLE | T_total / r (incluir censurados en T_total) |
| IC 90% lower | 2T / χ²(0.95, 2r+2) |
| Regla accionable | usar **IC_lower** para inventario y planificación |

| Ambiente | MTBF ref | R(1 año) |
|----------|----------|----------|
| Benigno | 1825 d | 82% |
| Moderado | 913 d | 67% |
| Severo | 365 d | 37% |
| Arena | 180 d | 13% |

---
*Notebook generado automáticamente por SIMBES para el Módulo 7 — Confiabilidad y MTBF.*  
*Versión: 1.0 · Fecha: 2026-03-15*